In [0]:

# ============================================================
# 05_MANAGER_CANDIDATE_SCORING
# Firmable - Rule-Based Manager Candidate Scoring
#
# Input:
#   workspace.firmable_v2_silver.manager_candidates
#
# Output:
#   workspace.firmable_v2_silver.manager_candidate_scores
#
# Important:
#   candidate_score is a rule-based score, NOT a probability.
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# 1. LOAD MANAGER CANDIDATES
# ============================================================

candidates = spark.table(
    "workspace.firmable_v2_silver.manager_candidates"
)

print("Candidate records loaded:", candidates.count())


# ============================================================
# 2. CALCULATE SENIORITY GAP
# A positive gap means the proposed manager is more senior.
# ============================================================

scored = candidates.withColumn(
    "seniority_gap",
    F.col("employee_seniority") - F.col("manager_seniority")
)


# ============================================================
# 3. STRUCTURAL RELATIONSHIP SCORE
# Stronger structural relationships receive higher scores.
# Functional Executive to Department Head receives a
# targeted adjustment from 45 to 49.
# ============================================================

scored = scored.withColumn(
    "relationship_fit_score",
    F.when(
        F.col("relationship_type") ==
        "Functional Executive to Department Head",
        49
    )
    .when(
        F.col("relationship_type").isin(
            "Team Lead to Individual Contributor",
            "Team Manager to Team Lead",
            "Department Head to Team Manager",
            "Division President to Department Head",
            "CEO to C-Suite Executive",
            "CEO to Division President",
            "CEO to Department Head - Flat"
        ),
        45
    )
    .when(
        (F.col("relationship_type") == "Department Head to Team Lead") &
        (F.col("org_structure") == "Flat"),
        45
    )
    .when(
        F.col("relationship_type") == "Team Manager to Individual Contributor",
        24
    )
    .when(
        F.col("relationship_type") == "Department Head to Team Lead",
        22
    )
    .when(
        F.col("relationship_type") == "CEO to Department Head - Fallback",
        18
    )
    .when(
        F.col("relationship_type") == "Department Head to Individual Contributor",
        10
    )
    .otherwise(0)
)


# ============================================================
# 4. TEAM ALIGNMENT SCORE
# Same non-null team receives 15 points.
# ============================================================

scored = scored.withColumn(
    "team_alignment_score",
    F.when(
        F.col("employee_team_id").isNotNull() &
        F.col("manager_team_id").isNotNull() &
        (F.col("employee_team_id") == F.col("manager_team_id")),
        15
    ).otherwise(0)
)


# ============================================================
# 5. DEPARTMENT ALIGNMENT SCORE
# Same non-null department receives 12 points.
# ============================================================

scored = scored.withColumn(
    "department_alignment_score",
    F.when(
        F.col("employee_department").isNotNull() &
        F.col("manager_department").isNotNull() &
        (F.col("employee_department") == F.col("manager_department")),
        12
    ).otherwise(0)
)


# ============================================================
# 6. BUSINESS UNIT ALIGNMENT SCORE
# Same non-null business unit receives 8 points.
# ============================================================

scored = scored.withColumn(
    "business_unit_alignment_score",
    F.when(
        F.col("employee_business_unit").isNotNull() &
        F.col("manager_business_unit").isNotNull() &
        (F.col("employee_business_unit") == F.col("manager_business_unit")),
        8
    ).otherwise(0)
)


# ============================================================
# 7. MANAGER AUTHORITY SCORE
# Management authority is supporting evidence.
# ============================================================

scored = scored.withColumn(
    "authority_score",
    F.when(F.col("manager_authority") == "People Manager", 8)
    .when(F.col("manager_authority") == "Potential People Manager", 5)
    .when(F.col("manager_authority") == "Executive", 4)
    .otherwise(0)
)


# ============================================================
# 8. SENIORITY SCORE
# Greater seniority gaps provide supporting evidence.
# ============================================================

scored = scored.withColumn(
    "seniority_score",
    F.when(F.col("seniority_gap").between(1, 2), 5)
    .when(F.col("seniority_gap").between(3, 4), 3)
    .when(F.col("seniority_gap").between(5, 6), 2)
    .when(F.col("seniority_gap") > 6, 1)
    .otherwise(0)
)


# ============================================================
# 9. CALCULATE FINAL CANDIDATE SCORE
# Maximum score is capped at 100.
# This is a rule-based score, not a probability.
# ============================================================

score_components = [
    "relationship_fit_score",
    "team_alignment_score",
    "department_alignment_score",
    "business_unit_alignment_score",
    "authority_score",
    "seniority_score"
]

scored = scored.withColumn(
    "candidate_score",
    F.least(
        F.lit(100),
        sum(F.col(column) for column in score_components)
    )
)


# ============================================================
# 10. RANK CANDIDATES FOR EACH EMPLOYEE
# Highest score is ranked first.
# Candidate priority, seniority gap and manager ID
# provide deterministic tie-breaking.
# ============================================================

ranking_window = (
    Window
    .partitionBy("company_id", "employee_id")
    .orderBy(
        F.col("candidate_score").desc(),
        F.col("candidate_priority").asc(),
        F.col("seniority_gap").asc(),
        F.col("candidate_manager_id").asc()
    )
)

scored = scored.withColumn(
    "candidate_rank",
    F.row_number().over(ranking_window)
)


# ============================================================
# 11. CALCULATE TOP AND SECOND-BEST CANDIDATE SCORES
# ============================================================

score_window = (
    Window
    .partitionBy("company_id", "employee_id")
    .orderBy(
        F.col("candidate_score").desc(),
        F.col("candidate_priority").asc(),
        F.col("seniority_gap").asc(),
        F.col("candidate_manager_id").asc()
    )
)

scored = (
    scored
    .withColumn(
        "top_candidate_score",
        F.first("candidate_score").over(score_window)
    )
    .withColumn(
        "second_candidate_score",
        F.lead("candidate_score", 1).over(score_window)
    )
)


# ============================================================
# 12. CALCULATE SCORE MARGIN
# A small margin indicates competing candidates with
# similar scores.
# ============================================================

scored = scored.withColumn(
    "score_margin",
    F.when(
        F.col("second_candidate_score").isNotNull(),
        F.col("top_candidate_score") -
        F.col("second_candidate_score")
    ).otherwise(None)
)


# ============================================================
# 13. FLAG AMBIGUOUS CANDIDATES
# Margin below 8 is considered ambiguous.
# A single candidate is not marked ambiguous.
# ============================================================

scored = scored.withColumn(
    "ambiguity_flag",
    F.when(
        F.col("score_margin").isNotNull() &
        (F.col("score_margin") < 8),
        True
    ).otherwise(False)
)


# ============================================================
# 14. PUBLISH SCORED CANDIDATES
# ============================================================

output_table = (
    "workspace.firmable_v2_silver.manager_candidate_scores"
)

(
    scored.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(output_table)
)

print("Candidate scoring table published:", output_table)


# ============================================================
# 15. VALIDATION CHECKS
# ============================================================

final_scores = spark.table(output_table)

total_candidates = final_scores.count()

duplicate_pairs = (
    final_scores
    .groupBy("company_id", "employee_id", "candidate_manager_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_scores = (
    final_scores
    .filter(
        F.col("candidate_score").isNull() |
        (F.col("candidate_score") < 0) |
        (F.col("candidate_score") > 100)
    )
    .count()
)

invalid_seniority_gaps = (
    final_scores
    .filter(
        F.col("seniority_gap").isNull() |
        (F.col("seniority_gap") <= 0)
    )
    .count()
)

invalid_margins = (
    final_scores
    .filter(
        F.col("score_margin").isNotNull() &
        (F.col("score_margin") < 0)
    )
    .count()
)

print("\n========== SCORING VALIDATION ==========")
print("Total candidates:", total_candidates)
print("Duplicate employee-manager pairs:", duplicate_pairs)
print("Invalid candidate scores:", invalid_scores)
print("Invalid seniority gaps:", invalid_seniority_gaps)
print("Invalid score margins:", invalid_margins)

if (
    duplicate_pairs == 0
    and invalid_scores == 0
    and invalid_seniority_gaps == 0
    and invalid_margins == 0
):
    print("RESULT: ALL SCORING VALIDATIONS PASSED")
else:
    raise Exception("Scoring validation failed. Review the results above.")


# ============================================================
# 16. SCORING SUMMARY
# ============================================================

print("\n========== SCORING SUMMARY ==========")

final_scores.select(
    F.count("*").alias("total_candidates"),
    F.round(F.avg("candidate_score"), 2).alias("average_candidate_score"),
    F.round(F.min("candidate_score"), 2).alias("minimum_candidate_score"),
    F.round(F.max("candidate_score"), 2).alias("maximum_candidate_score"),
    F.sum(
        F.when(F.col("ambiguity_flag") == True, 1).otherwise(0)
    ).alias("ambiguous_candidates")
).show(truncate=False)


# ============================================================
# 17. RELATIONSHIP SCORE DISTRIBUTION
# ============================================================

print("\n========== RELATIONSHIP SCORE DISTRIBUTION ==========")

final_scores.groupBy(
    "relationship_type",
    "relationship_fit_score"
).count().orderBy(
    "relationship_type"
).show(100, truncate=False)


# ============================================================
# 18. TOP CANDIDATE SCORE DISTRIBUTION
# ============================================================

print("\n========== TOP CANDIDATE SCORE DISTRIBUTION ==========")

(
    final_scores
    .filter(F.col("candidate_rank") == 1)
    .groupBy("candidate_score")
    .count()
    .orderBy("candidate_score")
    .show(100, truncate=False)
)

print("\nNOTE: Candidate scores are rule-based ranking scores,")
print("not calibrated probabilities or percentage confidence.")
print("Notebook 05 completed successfully.")
